# Generación de Datos - Sistema de Recomendaciones E-commerce
## Responsable: Gamaliel Castro
## Semana 1 - Big Data Project

Este notebook descarga la semilla MovieLens 25M, asigna puntajes
a las acciones de usuario y amplifica hasta 1 millón de usuarios.

In [1]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rand, expr, monotonically_increasing_id
import os
import zipfile

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


In [2]:
spark = SparkSession.builder \
    .appName("Generacion_Datos_MovieLens") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"✅ Spark iniciado correctamente - versión: {spark.version}")

26/05/06 22:42:38 WARN Utils: Your hostname, MacBook-Air-de-Gamaliel.local resolves to a loopback address: 127.0.0.1; using 192.168.100.93 instead (on interface en0)
26/05/06 22:42:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 22:42:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark iniciado correctamente - versión: 3.5.1


In [3]:
# Rutas base del proyecto
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW_DATA_PATH = os.path.join(PROJECT_ROOT, "data", "raw")
MOVIELENS_PATH = os.path.join(RAW_DATA_PATH, "movielens")

# Crear carpetas si no existen
os.makedirs(MOVIELENS_PATH, exist_ok=True)

print(f"📁 Raíz del proyecto: {PROJECT_ROOT}")
print(f"📁 Datos crudos: {RAW_DATA_PATH}")
print(f"📁 MovieLens: {MOVIELENS_PATH}")

📁 Raíz del proyecto: /Users/gamalielcastro/Documents/proyecto-ecommerce-bigdata
📁 Datos crudos: /Users/gamalielcastro/Documents/proyecto-ecommerce-bigdata/data/raw
📁 MovieLens: /Users/gamalielcastro/Documents/proyecto-ecommerce-bigdata/data/raw/movielens


In [4]:
import urllib.request

MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"
ZIP_PATH = os.path.join(RAW_DATA_PATH, "ml-25m.zip")

if not os.path.exists(ZIP_PATH):
    print("⬇️ Descargando MovieLens 25M (~250MB)...")
    urllib.request.urlretrieve(MOVIELENS_URL, ZIP_PATH)
    print("✅ Descarga completa")
else:
    print("✅ Archivo ya existe, omitiendo descarga")

print(f"📦 Tamaño: {os.path.getsize(ZIP_PATH) / 1024 / 1024:.1f} MB")

⬇️ Descargando MovieLens 25M (~250MB)...
✅ Descarga completa
📦 Tamaño: 249.8 MB


In [5]:
print("📦 Descomprimiendo MovieLens 25M...")

with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(RAW_DATA_PATH)

print("✅ Descompresión completa")
print("📁 Archivos disponibles:")
for f in os.listdir(os.path.join(RAW_DATA_PATH, "ml-25m")):
    size = os.path.getsize(os.path.join(RAW_DATA_PATH, "ml-25m", f)) / 1024 / 1024
    print(f"   - {f} ({size:.1f} MB)")

📦 Descomprimiendo MovieLens 25M...
✅ Descompresión completa
📁 Archivos disponibles:
   - links.csv (1.3 MB)
   - tags.csv (37.0 MB)
   - genome-tags.csv (0.0 MB)
   - ratings.csv (646.8 MB)
   - README.txt (0.0 MB)
   - genome-scores.csv (415.0 MB)
   - movies.csv (2.9 MB)


In [6]:
RATINGS_PATH = os.path.join(RAW_DATA_PATH, "ml-25m", "ratings.csv")
MOVIES_PATH = os.path.join(RAW_DATA_PATH, "ml-25m", "movies.csv")

# Cargar con Spark
ratings = spark.read.csv(RATINGS_PATH, header=True, inferSchema=True)
movies = spark.read.csv(MOVIES_PATH, header=True, inferSchema=True)

print(" RATINGS:")
print(f"   - Total registros: {ratings.count():,}")
print(f"   - Columnas: {ratings.columns}")

print("\n MOVIES:")
print(f"   - Total películas: {movies.count():,}")
print(f"   - Columnas: {movies.columns}")

print("\n Muestra de ratings:")
ratings.show(5)

 RATINGS:


[Stage 4:>                                                          (0 + 8) / 8]

   - Total registros: 25,000,095
   - Columnas: ['userId', 'movieId', 'rating', 'timestamp']

 MOVIES:
   - Total películas: 62,423
   - Columnas: ['movieId', 'title', 'genres']

 Muestra de ratings:
+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|    296|   5.0|1147880044|
|     1|    306|   3.5|1147868817|
|     1|    307|   5.0|1147868828|
|     1|    665|   5.0|1147878820|
|     1|    899|   3.5|1147868510|
+------+-------+------+----------+
only showing top 5 rows



In [7]:
from pyspark.sql.functions import when, lit

# Convertir rating numérico a tipo de evento y score base
# Según la tabla del proyecto:
# Reseña >= 4 estrellas = 6.0
# Reseña < 4 estrellas se trata como vista = 1.0
# Compra simulada para ratings = 5.0 → score 5.0

events = ratings.withColumn("event_type",
    when(col("rating") >= 4.0, lit("review_positive"))
    .otherwise(lit("view"))
).withColumn("base_score",
    when(col("rating") == 5.0, lit(6.0))
    .when(col("rating") >= 4.0, lit(3.0))
    .otherwise(lit(1.0))
)

print("📊 Distribución de eventos:")
events.groupBy("event_type").count().show()

print("📋 Muestra de eventos:")
events.show(5)

📊 Distribución de eventos:


+---------------+--------+
|     event_type|   count|
+---------------+--------+
|review_positive|12452811|
|           view|12547284|
+---------------+--------+

📋 Muestra de eventos:
+------+-------+------+----------+---------------+----------+
|userId|movieId|rating| timestamp|     event_type|base_score|
+------+-------+------+----------+---------------+----------+
|     1|    296|   5.0|1147880044|review_positive|       6.0|
|     1|    306|   3.5|1147868817|           view|       1.0|
|     1|    307|   5.0|1147868828|review_positive|       6.0|
|     1|    665|   5.0|1147878820|review_positive|       6.0|
|     1|    899|   3.5|1147868510|           view|       1.0|
+------+-------+------+----------+---------------+----------+
only showing top 5 rows



In [8]:
from pyspark.sql.functions import round as spark_round

# Usuarios originales en la semilla
original_users = ratings.select("userId").distinct().count()
print(f"👥 Usuarios originales: {original_users:,}")
print(f"🎯 Usuarios objetivo: 1,000,000")

# Amplificar repitiendo el dataset con nuevos userIds
amplified = events
for i in range(1, 7):  # 6 copias adicionales = ~1.1M usuarios
    copy = events.withColumn("userId", 
        (col("userId") + lit(i * 200000)).cast("integer")
    ).withColumn("base_score",
        spark_round(col("base_score") * (1 + (rand() - 0.5) * 0.1), 2)
    )
    amplified = amplified.union(copy)

# Verificar usuarios únicos
total_users = amplified.select("userId").distinct().count()
total_records = amplified.count()

print(f"\n✅ Amplificación completa:")
print(f"   - Usuarios únicos: {total_users:,}")
print(f"   - Total registros: {total_records:,}")

👥 Usuarios originales: 162,541
🎯 Usuarios objetivo: 1,000,000


[Stage 27:================================================>       (48 + 8) / 56]


✅ Amplificación completa:
   - Usuarios únicos: 1,137,787
   - Total registros: 175,000,665


In [9]:
from pyspark.sql.functions import sum as spark_sum

# Agregar scores por par usuario-producto
interactions = amplified \
    .groupBy("userId", "movieId") \
    .agg(spark_sum("base_score").alias("rating")) \
    .filter(col("rating") > 0)

total_interactions = interactions.count()
print(f"✅ Interacciones únicas usuario-producto: {total_interactions:,}")
print("\n📋 Muestra de interacciones:")
interactions.show(10)

✅ Interacciones únicas usuario-producto: 175,000,665

📋 Muestra de interacciones:


[Stage 38:>                                                         (0 + 1) / 1]

+------+-------+------+
|userId|movieId|rating|
+------+-------+------+
|     3|   5004|   3.0|
|     3|   6754|   3.0|
|     3|  35836|   1.0|
|     3|  87232|   3.0|
|     4|   3624|   1.0|
|     4| 122882|   6.0|
|     5|   1246|   6.0|
|     6|   2028|   1.0|
|     8|   1777|   1.0|
|    11|  48043|   3.0|
+------+-------+------+
only showing top 10 rows



In [10]:
from datetime import date

RUN_DATE = "2026-05-10"
OUTPUT_PATH = os.path.join(PROJECT_ROOT, "data", "raw", f"interactions_run_date={RUN_DATE}")

print(f"💾 Guardando interacciones en: {OUTPUT_PATH}")

interactions.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(OUTPUT_PATH)

print(f"✅ Guardado completo")

# Verificar archivos generados
files = os.listdir(OUTPUT_PATH)
csv_files = [f for f in files if f.endswith(".csv")]
print(f"📁 Archivos CSV generados: {len(csv_files)}")

💾 Guardando interacciones en: /Users/gamalielcastro/Documents/proyecto-ecommerce-bigdata/data/raw/interactions_run_date=2026-05-10


[Stage 41:======================================================> (33 + 1) / 34]

✅ Guardado completo
📁 Archivos CSV generados: 34


In [14]:
from pyspark.sql.functions import lit, rand as spark_rand

CATALOG_PATH = os.path.join(PROJECT_ROOT, "data", "raw", f"catalog_run_date={RUN_DATE}")

print(f"💾 Guardando catálogo en: {CATALOG_PATH}")

catalog = movies.withColumn("is_active", lit(True)) \
    .withColumn("stock", (spark_rand() * 100 + 1).cast("integer")) \
    .withColumn("price", spark_round(spark_rand() * 50 + 5, 2)) \
    .withColumn("category", col("genres"))

catalog.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(CATALOG_PATH)

print("✅ Catálogo guardado")
catalog.show(5)

💾 Guardando catálogo en: /Users/gamalielcastro/Documents/proyecto-ecommerce-bigdata/data/raw/catalog_run_date=2026-05-10
✅ Catálogo guardado
+-------+--------------------+--------------------+---------+-----+-----+--------------------+
|movieId|               title|              genres|is_active|stock|price|            category|
+-------+--------------------+--------------------+---------+-----+-----+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|     true|   35|23.08|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|     true|   91|31.61|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|     true|   29|13.91|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|     true|   83|21.37|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|     true|   60|48.49|              Comedy|
+-------+--------------------+--------------------+---------+-----+-----+--------------------+
only

In [15]:
print("=" * 50)
print("📊 RESUMEN FINAL DE GENERACIÓN DE DATOS")
print("=" * 50)

# Contar archivos generados
interactions_files = len([f for f in os.listdir(OUTPUT_PATH) if f.endswith(".csv")])
catalog_files = len([f for f in os.listdir(CATALOG_PATH) if f.endswith(".csv")])

# Tamaño en disco
def get_folder_size(path):
    total = 0
    for f in os.listdir(path):
        fp = os.path.join(path, f)
        if os.path.isfile(fp):
            total += os.path.getsize(fp)
    return total / 1024 / 1024 / 1024  # GB

interactions_size = get_folder_size(OUTPUT_PATH)
catalog_size = get_folder_size(CATALOG_PATH)

print(f"👥 Usuarios únicos:         {total_users:>15,}")
print(f"🎬 Películas en catálogo:   {movies.count():>15,}")
print(f"🔗 Interacciones totales:   {total_interactions:>15,}")
print(f"📁 Archivos interacciones:  {interactions_files:>15}")
print(f"📁 Archivos catálogo:       {catalog_files:>15}")
print(f"💾 Tamaño interacciones:    {interactions_size:>14.2f} GB")
print(f"💾 Tamaño catálogo:         {catalog_size:>14.4f} GB")
print("=" * 50)
print("✅ Dataset listo para la Fase 1 (ETL)")

📊 RESUMEN FINAL DE GENERACIÓN DE DATOS
👥 Usuarios únicos:               1,137,787
🎬 Películas en catálogo:            62,423
🔗 Interacciones totales:       175,000,665
📁 Archivos interacciones:               34
📁 Archivos catálogo:                     1
💾 Tamaño interacciones:              2.79 GB
💾 Tamaño catálogo:                 0.0044 GB
✅ Dataset listo para la Fase 1 (ETL)


In [1]:
from minio import Minio

# Conexión a MinIO de Blanquita
client = Minio(
    "10.150.6.118:9000",
    access_key="admin",
    secret_key="admin12345",
    secure=False
)

print("✅ Conexión a MinIO establecida")
print(f"📦 Buckets disponibles: {[b.name for b in client.list_buckets()]}")

✅ Conexión a MinIO establecida
📦 Buckets disponibles: ['raw']


In [3]:
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
RUN_DATE = "2026-05-10"
OUTPUT_PATH = os.path.join(PROJECT_ROOT, "data", "raw", f"interactions_run_date={RUN_DATE}")
CATALOG_PATH = os.path.join(PROJECT_ROOT, "data", "raw", f"catalog_run_date={RUN_DATE}")

print(f"✅ Rutas redefinidas")
print(f"   interactions: {OUTPUT_PATH}")
print(f"   catalog:      {CATALOG_PATH}")

✅ Rutas redefinidas
   interactions: /Users/gamalielcastro/Documents/proyecto-ecommerce-bigdata/data/raw/interactions_run_date=2026-05-10
   catalog:      /Users/gamalielcastro/Documents/proyecto-ecommerce-bigdata/data/raw/catalog_run_date=2026-05-10


In [4]:
import glob

def upload_folder_to_minio(local_path, bucket, prefix):
    """Sube todos los CSVs de una carpeta al bucket de MinIO"""
    archivos = glob.glob(os.path.join(local_path, "*.csv"))
    total = len(archivos)
    
    for i, filepath in enumerate(archivos, 1):
        filename = os.path.basename(filepath)
        object_name = f"{prefix}/{filename}"
        client.fput_object(bucket, object_name, filepath)
        print(f"  [{i}/{total}] ✅ {filename}")
    
    return total

print("🚀 Iniciando subida a MinIO...\n")

# Subir interacciones
print("📁 Subiendo interactions_run_date=2026-05-10...")
total1 = upload_folder_to_minio(
    OUTPUT_PATH,
    "raw",
    "interactions_run_date=2026-05-10"
)

# Subir catálogo
print(f"\n📁 Subiendo catalog_run_date=2026-05-10...")
total2 = upload_folder_to_minio(
    CATALOG_PATH,
    "raw",
    "catalog_run_date=2026-05-10"
)

print(f"\n✅ Subida completa")
print(f"   - Archivos de interacciones: {total1}")
print(f"   - Archivos de catálogo: {total2}")

🚀 Iniciando subida a MinIO...

📁 Subiendo interactions_run_date=2026-05-10...
  [1/34] ✅ part-00016-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [2/34] ✅ part-00001-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [3/34] ✅ part-00022-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [4/34] ✅ part-00017-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [5/34] ✅ part-00000-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [6/34] ✅ part-00023-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [7/34] ✅ part-00018-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [8/34] ✅ part-00014-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [9/34] ✅ part-00020-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [10/34] ✅ part-00003-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [11/34] ✅ part-00019-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv


S3Error: S3 operation failed; code: RequestTimeTooSkewed, message: The difference between the request time and the server's time is too large., resource: /raw/interactions_run_date=2026-05-10/part-00015-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv, request_id: 18AEE6BCB3A104E9, host_id: dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8

In [5]:
import glob

def upload_folder_to_minio(local_path, bucket, prefix):
    archivos = sorted(glob.glob(os.path.join(local_path, "*.csv")))
    total = len(archivos)
    
    for i, filepath in enumerate(archivos, 1):
        filename = os.path.basename(filepath)
        object_name = f"{prefix}/{filename}"
        try:
            client.stat_object(bucket, object_name)
            print(f"  [{i}/{total}] ⏭️ Ya existe: {filename}")
            continue
        except:
            pass
        client.fput_object(bucket, object_name, filepath)
        print(f"  [{i}/{total}] ✅ {filename}")
    
    return total

print("🚀 Reiniciando subida (saltando archivos ya subidos)...\n")

print("📁 Subiendo interactions_run_date=2026-05-10...")
total1 = upload_folder_to_minio(OUTPUT_PATH, "raw", "interactions_run_date=2026-05-10")

print(f"\n📁 Subiendo catalog_run_date=2026-05-10...")
total2 = upload_folder_to_minio(CATALOG_PATH, "raw", "catalog_run_date=2026-05-10")

print(f"\n✅ Subida completa")
print(f"   - Archivos de interacciones: {total1}")
print(f"   - Archivos de catálogo:      {total2}")

🚀 Reiniciando subida (saltando archivos ya subidos)...

📁 Subiendo interactions_run_date=2026-05-10...
  [1/34] ⏭️ Ya existe: part-00000-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [2/34] ⏭️ Ya existe: part-00001-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [3/34] ✅ part-00002-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [4/34] ⏭️ Ya existe: part-00003-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [5/34] ✅ part-00004-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [6/34] ✅ part-00005-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [7/34] ✅ part-00006-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [8/34] ✅ part-00007-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [9/34] ✅ part-00008-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [10/34] ✅ part-00009-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [11/34] ✅ part-00010-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [12/34] ✅ part-00011-02336cbd-84b6-4480-9b4f-3fa97daf25be-c000.csv
  [13/34] ✅ part-00012-02336cbd-84b6-4480-